## Automated Financial Market Ingestion Engine
#### Author: Sachin Kumar B
#### Date: 27/07/2026

In [1]:
# Step 1: Imports
import os
import json
import time
import random
import glob
import threading
import requests
import pandas as pd
from datetime import datetime
from flask import Flask, jsonify

In [5]:
# Step 2: Start Background Mock API Server
app = Flask(__name__)

@app.route('/v1/market/ticks/<symbol>', methods=['GET'])
def get_ticks(symbol):
    return jsonify({
        "symbol": symbol.upper(),
        "timestamp": int(time.time()),
        "price": round(random.uniform(150.0, 180.0), 2),
        "volume": random.randint(100, 5000),
        "status": "OK"
    })

def run_server():
    app.run(port=5001, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1)
print("Mock Market Server listening on http://localhost:5001")

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


Mock Market Server listening on http://localhost:8000


In [6]:
# Step 3: Streaming Ingestion Function
STORAGE_DIR = "data/market_ticks"
os.makedirs(STORAGE_DIR, exist_ok=True)

def fetch_and_store_ticks(symbol="AAPL", iterations=5, delay_seconds=1):
    print(f"Starting streaming ingestion for {symbol}...")
    for i in range(iterations):
        response = requests.get(f"http://localhost:5001/v1/market/ticks/{symbol}")
        if response.status_code == 200:
            data = response.json()
            ts = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            filename = f"{STORAGE_DIR}/{symbol}_tick_{ts}.json"
            
            with open(filename, 'w') as f:
                json.dump(data, f, indent=2)
                
            print(f"[{i+1}/{iterations}] Saved {symbol} price: ${data['price']} -> {filename}")
        time.sleep(delay_seconds)

fetch_and_store_ticks("AAPL", iterations=5, delay_seconds=1)

Starting streaming ingestion for AAPL...


127.0.0.1 - - [27/Jul/2026 14:47:22] "GET /v1/market/ticks/AAPL HTTP/1.1" 200 -


[1/5] Saved AAPL price: $150.85 -> data/market_ticks/AAPL_tick_20260727_144722_412547.json


127.0.0.1 - - [27/Jul/2026 14:47:25] "GET /v1/market/ticks/AAPL HTTP/1.1" 200 -


[2/5] Saved AAPL price: $160.37 -> data/market_ticks/AAPL_tick_20260727_144725_493815.json


...

[3/5] Saved AAPL price: $175.77 -> data/market_ticks/AAPL_tick_20260727_144728_542318.json


127.0.0.1 - - [27/Jul/2026 14:47:31] "GET /v1/market/ticks/AAPL HTTP/1.1" 200 -


[4/5] Saved AAPL price: $173.71 -> data/market_ticks/AAPL_tick_20260727_144731_615948.json


127.0.0.1 - - [27/Jul/2026 14:47:34] "GET /v1/market/ticks/AAPL HTTP/1.1" 200 -


[5/5] Saved AAPL price: $173.25 -> data/market_ticks/AAPL_tick_20260727_144734_679468.json


In [7]:
# Step 4: Analyze Ingested Ticks with Pandas
json_files = glob.glob(f"{STORAGE_DIR}/AAPL_tick_*.json")
tick_data = []

for fpath in json_files:
    with open(fpath, 'r') as f:
        tick_data.append(json.load(f))

df_ticks = pd.DataFrame(tick_data)
df_ticks['timestamp'] = pd.to_datetime(df_ticks['timestamp'], unit='s')
df_ticks.sort_values(by='timestamp', ascending=False, inplace=True)

print("\nIngested Ticks Data Summary:")
print(df_ticks.to_string(index=False))
print(f"\nAverage AAPL Price: ${df_ticks['price'].mean():.2f}")


Ingested Ticks Data Summary:
 price status symbol           timestamp  volume
173.25     OK   AAPL 2026-07-27 09:17:34    1093
173.71     OK   AAPL 2026-07-27 09:17:31    4543
175.77     OK   AAPL 2026-07-27 09:17:28    2945
160.37     OK   AAPL 2026-07-27 09:17:25     731
150.85     OK   AAPL 2026-07-27 09:17:22    3613
152.49     OK   AAPL 2026-07-27 09:16:43     884
179.18     OK   AAPL 2026-07-27 09:16:40     820
167.96     OK   AAPL 2026-07-27 09:16:37    1020
159.70     OK   AAPL 2026-07-27 09:16:34    1107
168.21     OK   AAPL 2026-07-27 09:16:31    1973

Average AAPL Price: $166.15


127.0.0.1 - - [27/Jul/2026 14:48:01] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [27/Jul/2026 14:48:02] "GET /favicon.ico HTTP/1.1" 404 -
